# OjoVial -- Prueba de Camara
Deteccion en tiempo real con webcam. Presiona **q** para salir.

In [ ]:
import os
import urllib.request
import cv2
import numpy as np
from ultralytics import YOLO
import time

In [ ]:
MODEL_PATH = "models/best.pt"
MODEL_URL  = "https://github.com/TomaX04/OjoVial2/releases/download/v1.0/best.pt"

if not os.path.exists(MODEL_PATH):
    os.makedirs("models", exist_ok=True)
    print("Descargando modelo desde GitHub Releases...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"Modelo guardado en {MODEL_PATH}")
else:
    print(f"Modelo encontrado en {MODEL_PATH}")

In [ ]:
CLASSES = {
    0: ("Pare (Stop)",             (68, 68, 255)),
    1: ("Ceda el Paso (Yield)",    (0, 215, 255)),
    2: ("No Right Turn",           (0, 140, 255)),
    3: ("No Left Turn",            (255, 136, 68)),
    4: ("Go Straight",             (68, 255, 68)),
    5: ("Speed Limit 40",          (187, 187, 187)),
    6: ("Speed Limit 60",          (153, 153, 153)),
    7: ("No Parking",              (204, 51, 153)),
}

IMG_SIZE       = 640

In [ ]:
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {model.names}")

In [ ]:
def draw_detections(frame, results):
    for r in results:
        boxes = r.boxes
        if boxes is None:
            continue
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf  = float(box.conf[0])
            clsid = int(box.cls[0])
            name, color = CLASSES.get(clsid, (f"Class {clsid}", (255, 255, 255)))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f"{name} {conf * 100:.1f}%"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)
            cv2.putText(frame, label, (x1 + 3, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return frame

CAM_INDEX = 2
cap = cv2.VideoCapture(CAM_INDEX)

if not cap.isOpened():
    print(f"ERROR: No se pudo abrir camara {CAM_INDEX}")
else:
    print(f"Camara {CAM_INDEX} abierta. Presiona q para salir.")

    fps_time = time.time()
    fps_count = 0
    fps_display = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error al leer frame")
            break

        results = model.predict(
            source=frame, conf=0.65, iou=0.50,
            imgsz=IMG_SIZE, verbose=False,
        )
        frame = draw_detections(frame, results)

        fps_count += 1
        if time.time() - fps_time >= 1.0:
            fps_display = fps_count
            fps_count = 0
            fps_time = time.time()

        cv2.putText(frame, f"FPS: {fps_display}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.imshow("OjoVial", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Camara liberada.")